In [1]:
import pandas as pd
from scipy.stats import zscore
import numpy as np

In [3]:
# Generate a mapping of possible unique values for each column in the DataFrame
df_raw = pd.read_csv('train.csv')

df = df_raw.drop(columns=['Utilities', 'Street', 'Condition2', 'RoofMatl', 'Heating', 'PoolQC', 'Exterior1st', 'SaleType'])

mssubclass_map = {
    20: '1-STORY 1946 & NEWER ALL STYLES',
    30: '1-STORY 1945 & OLDER',
    40: '1-STORY W/FINISHED ATTIC ALL AGES',
    45: '1-1/2 STORY - UNFINISHED ALL AGES',
    50: '1-1/2 STORY FINISHED ALL AGES',
    60: '2-STORY 1946 & NEWER',
    70: '2-STORY 1945 & OLDER',
    75: '2-1/2 STORY ALL AGES',
    80: 'SPLIT OR MULTI-LEVEL',
    85: 'SPLIT FOYER',
    90: 'DUPLEX - ALL STYLES AND AGES',
    120: '1-STORY PUD (Planned Unit Development) - 1946 & NEWER',
    150: '1-1/2 STORY PUD - ALL AGES',
    160: '2-STORY PUD - 1946 & NEWER',
    180: 'PUD - MULTILEVEL - INCL SPLIT LEV/FOYER',
    190: '2 FAMILY CONVERSION - ALL STYLES AND AGES',
}
df['MSSubClass'] = df['MSSubClass'].map(mssubclass_map)

to_str_columns = ['GarageYrBlt']
df[to_str_columns] = df[to_str_columns].astype(str)

for column in df.columns:
    if df[column].dtype == 'object' or df[column].dtype.name == 'str':
        df.fillna({column: 'None'}, inplace=True)

possible_values_map = {}
for column in df.columns:
    if df[column].dtype == 'object' or df[column].dtype.name == 'str':
      unique_values = ','.join(sorted(list(df[column].unique())))
      if unique_values not in possible_values_map.keys():
          possible_values_map[unique_values] = [column]
      else:
          possible_values_map[unique_values].append(column)
          
with open("possible_values_map.txt", "w") as f:
    for unique_values, columns in possible_values_map.items():
        f.write(f"\nColumns: {columns}\n")
        f.write(f"'{unique_values}' = {{\n\t{unique_values.replace(',', ': ,\n\t')}: ,\n}}")
        f.write("\n")

In [4]:
# Defines function to parse the cleaned map to dict

import ast
import re

def parse_mappings(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    result = {}

    # Match each Columns declaration and its mapping block
    pattern = r"Columns:\s*(\[.*?\])\s*'[^']*':\s*\{(.*?)\}"

    for columns_str, mapping_str in re.findall(
        pattern, text, flags=re.DOTALL
    ):
        columns = ast.literal_eval(columns_str)

        # Parse key:value pairs
        mapping = {}
        for key, value in re.findall(
            r"([^,\s]+)\s*:\s*(\d+)\s*,?", mapping_str
        ):
            mapping[key] = int(value)

        for column in columns:
            result[column] = mapping

    return result

In [7]:
# - Faça o download do dataset e carregue-o usando a biblioteca Pandas;
df_raw = pd.read_csv('train.csv')

# - Remova as colunas: Utilities, Street, Condition2, RoofMatl, Heating, PoolQC, Exterior1st e SaleType.
df = df_raw.drop(columns=['Utilities', 'Street', 'Condition2', 'RoofMatl', 'Heating', 'PoolQC', 'Exterior1st', 'SaleType'])

# - Trate os NA ("None") do dataset. No caso de ser um campo categorico trate isso como mais uma categoria, se for um campo númerico utilize a média da coluna.
mssubclass_map = {
    20: '1-STORY 1946 & NEWER ALL STYLES',
    30: '1-STORY 1945 & OLDER',
    40: '1-STORY W/FINISHED ATTIC ALL AGES',
    45: '1-1/2 STORY - UNFINISHED ALL AGES',
    50: '1-1/2 STORY FINISHED ALL AGES',
    60: '2-STORY 1946 & NEWER',
    70: '2-STORY 1945 & OLDER',
    75: '2-1/2 STORY ALL AGES',
    80: 'SPLIT OR MULTI-LEVEL',
    85: 'SPLIT FOYER',
    90: 'DUPLEX - ALL STYLES AND AGES',
    120: '1-STORY PUD (Planned Unit Development) - 1946 & NEWER',
    150: '1-1/2 STORY PUD - ALL AGES',
    160: '2-STORY PUD - 1946 & NEWER',
    180: 'PUD - MULTILEVEL - INCL SPLIT LEV/FOYER',
    190: '2 FAMILY CONVERSION - ALL STYLES AND AGES',
}
df['MSSubClass'] = df['MSSubClass'].map(mssubclass_map)

to_str_columns = ['GarageYrBlt']
df[to_str_columns] = df[to_str_columns].astype(str)


for column in df.columns:
    if df[column].dtype == 'object' or df[column].dtype.name == 'str':
        df.fillna({column: 'None'}, inplace=True)
    else:
        df.fillna({column: df[column].mean()}, inplace=True)

# - Calcule o z-score para todas as colunas numericas, não categoricas, e aplique essa transformação para todos os elementos da coluna.
z_columns = df.select_dtypes(include=["number"]).columns.drop(['Id', 'SalePrice'])
df[z_columns] = df[z_columns].apply(zscore, ddof=0)
    

# - Transforme as colunas ordinais em números (Ex: {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}; )
mappings = parse_mappings("possible_values_map_clean.txt")
for column, mapping in mappings.items():
    df[column] = df[column].map(mapping)
    
# - Calcule o one-hot encoding para as colunas categóricas.
### - One-hot encoding introduces multicollinearity, but we retain all categories since the data is intended for neural networks.
categorical_columns = list(df.select_dtypes(include=["object", "category", "string"]).columns)
df = pd.get_dummies(df, columns=categorical_columns, dtype=int)

# - Para a coluna de SalePrice faça ln(z-score(...)) dos valores da coluna.
df["SalePrice"] = np.asarray(zscore(np.log(df["SalePrice"]), ddof=0))

df.to_csv('train_prepared.csv', index=False)